# Laboratório 3 – Alinhamento, Homografia 2D e Mosaico
# Visão Computacional — 2026.2

---

**Autores:**
 - João Victor de Castro Carvalho RA 11202420287
 - Mario Guerreiro Cerconvis RA 11202421459
 - Marco Antonio Silva da Mata 11202320279
 - João Victor Gonçalves Nascimento RA 11202420821

**Data de realização dos experimentos:** 28/06/2026

**Data de publicação do relatório:** 28/06/2026

---

## 1. Introdução

Este relatório apresenta os estudos e experimentos realizados no Laboratório 3 da disciplina Visão Computacional, cujo tema central é o **alinhamento de imagens bidimensionais**, a **estimação de homografia 2D** e a construção de **mosaicos de imagens** (Image Stitching).

O alinhamento de imagens é uma etapa essencial em diversas aplicações de visão computacional e robótica, como:
- Construção de panoramas fotográficos
- Mapeamento aéreo por drones
- Navegação autônoma baseada em visão (SLAM visual)
- Estabilização de vídeo
- Realidade aumentada
- Inspeção industrial e sensoriamento remoto

Neste laboratório, estudamos os fundamentos matemáticos da homografia 2D, analisamos o impacto de outliers no método de Mínimos Quadrados Tradicional, implementamos o algoritmo RANSAC para filtragem robusta de correspondências e desenvolvemos um pipeline completo de Image Stitching.

O relatório está organizado da seguinte forma:
1. **Fundamentação Teórica** — Feature Matching, Homografia 2D, RANSAC e Image Stitching
2. **Procedimentos Experimentais** — detalhamento dos experimentos realizados
3. **Análise e Discussão** — análise dos resultados e respostas às questões propostas
4. **Conclusões** — síntese dos aprendizados
5. **Referências** — fontes consultadas

## 2. Fundamentação Teórica

### 2.1 Feature Matching (Correspondência de Características)

O Feature Matching é o processo de encontrar correspondências entre pontos característicos (features) detectados em duas ou mais imagens. Ele é a base para diversas tarefas de visão computacional, como estimação de movimento, reconstrução 3D e alinhamento de imagens.

O pipeline típico de Feature Matching consiste em três etapas:

1. **Detecção de keypoints:** algoritmos como SIFT, SURF ou ORB identificam pontos de interesse na imagem que são estáveis sob transformações geométricas.

2. **Cálculo de descritores:** para cada keypoint, um vetor descritor é gerado, capturando a aparência local da região ao redor do ponto. No caso do SIFT, o descritor possui **128 dimensões**.

3. **Matching:** os descritores de uma imagem são comparados com os da outra usando métricas de distância (tipicamente a distância Euclidiana L2). O matcher pode ser do tipo Brute-Force (BFMatcher) ou baseado em árvores KD (FLANN).

#### Teste de Razão de Lowe

Um problema comum no matching é a presença de correspondências ambíguas. Lowe (2004) propôs um teste simples e eficaz: para cada keypoint da primeira imagem, encontram-se os dois descritores mais próximos na segunda imagem. Se a razão entre a distância do melhor e do segundo melhor match for menor que um limiar (tipicamente 0.75), a correspondência é aceita:

$$\frac{d(m_1)}{d(m_2)} < 0.75$$

Isso garante que apenas correspondências com alto grau de distinção sejam mantidas, eliminando matches ambíguos.

---

### 2.2 Homografia 2D

A homografia é uma transformação projetiva $3 \times 3$ que mapeia pontos de um plano em uma imagem para os pontos correspondentes em outra imagem. Ela é representada pela matriz $H$:

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} \sim \begin{bmatrix} h_{00} & h_{01} & h_{02} \\ h_{10} & h_{11} & h_{12} \\ h_{20} & h_{21} & h_{22} \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

onde $(x, y)$ são coordenadas na primeira imagem e $(x', y')$ na segunda. O símbolo $\sim$ indica igualdade a menos de um fator de escala.

A matriz $H$ possui 9 elementos, mas como é definida a menos de escala (normalmente $h_{22} = 1$), ela tem **8 graus de liberdade**. Como cada par de pontos correspondentes fornece 2 equações lineares independentes, são necessários no mínimo **4 pares de pontos** para calcular $H$.

#### Mínimos Quadrados Tradicionais

Quando temos mais de 4 pares de pontos, o sistema é sobredeterminado e pode ser resolvido por Mínimos Quadrados (Least Squares). No OpenCV, isso corresponde a `cv2.findHomography(src, dst, method=0)`. O problema é que este método trata todos os pontos igualmente se houver outliers (correspondências incorretas), eles distorcem significativamente a estimação da homografia.

---

### 2.3 RANSAC (Random Sample Consensus)

O RANSAC, proposto por Fischler e Bolles (1981), é um método iterativo para estimação robusta de parâmetros de modelos matemáticos na presença de outliers. Seu funcionamento é:

1. **Amostragem aleatória:** seleciona-se aleatoriamente o número mínimo de pontos necessário para ajustar o modelo (4 pares para homografia).

2. **Estimação do modelo:** calcula-se a homografia $H$ a partir da amostra.

3. **Avaliação de consenso:** projeta-se todos os pontos usando $H$ e conta-se quantos são **inliers** (erro de reprojeção menor que um limiar, tipicamente 5 pixels).

4. **Iteração:** repete-se o processo várias vezes, mantendo o modelo com o maior número de inliers.

5. **Refinamento:** ao final, a homografia é recalculada usando apenas os inliers do melhor modelo.

No OpenCV, utiliza-se `cv2.findHomography(src, dst, cv2.RANSAC, reproj_threshold)`. A vantagem do RANSAC é que ele é capaz de produzir estimativas precisas mesmo quando uma fração significativa dos dados é composta por outliers.

---

### 2.4 Image Stitching (Costura de Imagens)

Image Stitching é o processo de combinar múltiplas imagens com sobreposição em um único panorama. O pipeline clássico consiste em:

1. **Detecção de features** em cada imagem (SIFT, ORB, etc.)
2. **Feature matching** entre pares de imagens adjacentes
3. **Estimação da homografia** entre cada par de imagens
4. **Warping (projeção):** transformação de uma imagem para o sistema de coordenadas da outra usando `cv2.warpPerspective()`
5. **Composição do mosaico:** sobreposição das imagens projetadas em um canvas comum

A qualidade do mosaico depende da precisão da homografia estimada. Com Mínimos Quadrados clássicos, outliers podem causar desalinhamento severo. Com RANSAC, a estimação é mais robusta, produzindo mosaicos com melhor alinhamento.

Um problema comum no mosaico final é a presença de **artefatos visuais** na zona de junção: linhas de corte visíveis, diferenças de iluminação e efeito fantasma (ghosting). Técnicas de pós-processamento como **Multi-band Blending** (Burt & Adelson, 1983) e **Feathering** são usadas para suavizar essas transições.

## 3. Procedimentos Experimentais

### 3.0 Configuração do Ambiente

Primeiramente, importamos as bibliotecas necessárias e configuramos o ambiente de visualização.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Configuração para exibir imagens no notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

### 3.1 Função Auxiliar — Pipeline de Homografia e Mosaico

Para evitar repetição de código entre os experimentos A.1 e A.2, definimos uma função que encapsula todo o pipeline: detecção SIFT, matching com teste de Lowe, estimação de homografia (Mínimos Quadrados e RANSAC) e costura do mosaico.

In [ ]:
def pipeline_homografia_mosaico(path_img1, path_img2, titulo_experimento):
    """
    Pipeline completo de homografia e mosaico.
    Retorna as matrizes H_ls, H_ransac, contagens de inliers/outliers
    e salva as figuras geradas.
    """
    # -----------------------------------------------------------------
    # Carregamento das imagens
    # -----------------------------------------------------------------
    img1 = cv2.imread(path_img1)
    img2 = cv2.imread(path_img2)

    if img1 is None or img2 is None:
        raise FileNotFoundError(
            f"Verifique se as imagens foram salvas corretamente:\n"
            f"  img1: {path_img1}\n  img2: {path_img2}"
        )

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # -----------------------------------------------------------------
    # PASSO 1: Detecção SIFT e correspondência (matching)
    # -----------------------------------------------------------------
    sift = cv2.SIFT_create()

    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    bf = cv2.BFMatcher()
    matches_brutos = bf.knnMatch(des1, des2, k=2)

    # Teste de razão de Lowe
    bons_matches = []
    for m, n in matches_brutos:
        if m.distance < 0.75 * n.distance:
            bons_matches.append(m)

    src_pts = np.float32([kp1[m.queryIdx].pt for m in bons_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in bons_matches]).reshape(-1, 1, 2)

    print(f"=== {titulo_experimento} ===")
    print(f"Keypoints detectados: img1={len(kp1)}, img2={len(kp2)}")
    print(f"Total de correspondências validadas (pós-Lowe): {len(bons_matches)}")

    # -----------------------------------------------------------------
    # PASSO 2: Homografia via Mínimos Quadrados Tradicionais (method=0)
    # -----------------------------------------------------------------
    H_ls, status_ls = cv2.findHomography(src_pts, dst_pts, method=0)

    print("\n--- Matriz de Homografia via Mínimos Quadrados Clássicos ---")
    print(H_ls)

    # -----------------------------------------------------------------
    # PASSO 3: Homografia via RANSAC
    # -----------------------------------------------------------------
    reproj_threshold = 5.0
    H_ransac, mask_ransac = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, reproj_threshold)

    print("\n--- Matriz de Homografia via RANSAC ---")
    print(H_ransac)

    inliers_count = int(np.sum(mask_ransac))
    outliers_count = len(mask_ransac) - inliers_count
    total_matches = len(bons_matches)
    pct_outliers = (1 - inliers_count / total_matches) * 100

    print(f"\nResultado RANSAC: {inliers_count} Inliers | {outliers_count} Outliers")
    print(f"Porcentagem de Outliers: {pct_outliers:.2f}%")

    # -----------------------------------------------------------------
    # Visualização das correspondências (inliers do RANSAC)
    # -----------------------------------------------------------------
    img_matches = cv2.drawMatches(
        img1, kp1, img2, kp2, bons_matches, None,
        matchColor=(0, 255, 0), singlePointColor=None,
        matchesMask=mask_ransac.ravel().tolist(), flags=2
    )

    plt.figure(figsize=(14, 6))
    plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
    plt.title(f"{titulo_experimento} — Inliers RANSAC ({inliers_count}/{total_matches})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # PASSO 4: Costura de Imagens (Image Stitching)
    # -----------------------------------------------------------------
    width = img1.shape[1] + img2.shape[1]
    height = max(img1.shape[0], img2.shape[0])

    img1_warped_ls = cv2.warpPerspective(img1, H_ls, (width, height))
    img1_warped_ransac = cv2.warpPerspective(img1, H_ransac, (width, height))

    mosaico_ls = img1_warped_ls.copy()
    mosaico_ls[0:img2.shape[0], 0:img2.shape[1]] = img2

    mosaico_ransac = img1_warped_ransac.copy()
    mosaico_ransac[0:img2.shape[0], 0:img2.shape[1]] = img2

    fig, ax = plt.subplots(2, 1, figsize=(14, 10))
    ax[0].imshow(cv2.cvtColor(mosaico_ls, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"{titulo_experimento} — Mosaico Mínimos Quadrados Tradicionais")
    ax[0].axis('off')

    ax[1].imshow(cv2.cvtColor(mosaico_ransac, cv2.COLOR_BGR2RGB))
    ax[1].set_title(f"{titulo_experimento} — Mosaico RANSAC Robusto")
    ax[1].axis('off')

    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # Análise numérica das matrizes
    # -----------------------------------------------------------------
    print("\n--- Comparação dos elementos de translação ---")
    diff_h02 = abs(H_ls[0, 2] - H_ransac[0, 2])
    diff_h12 = abs(H_ls[1, 2] - H_ransac[1, 2])
    print(f"|h02_LS - h02_RANSAC| = |{H_ls[0,2]:.4f} - {H_ransac[0,2]:.4f}| = {diff_h02:.4f} pixels")
    print(f"|h12_LS - h12_RANSAC| = |{H_ls[1,2]:.4f} - {H_ransac[1,2]:.4f}| = {diff_h12:.4f} pixels")

    return {
        'H_ls': H_ls, 'H_ransac': H_ransac,
        'inliers': inliers_count, 'outliers': outliers_count,
        'total_matches': total_matches, 'pct_outliers': pct_outliers,
        'mosaico_ransac': mosaico_ransac, 'mosaico_ls': mosaico_ls
    }

### 3.2 Parte 2A.1 — Experimento com Pessoa

Neste experimento, tiramos duas fotos de uma cena contendo uma **pessoa**, com sobreposição de pelo menos 50% entre elas. A câmera foi ligeiramente rotacionada entre as capturas.

In [ ]:
# =====================================================================
# EXPERIMENTO A.1 — Pessoa
# Substitua pelos nomes dos seus arquivos de imagem
# =====================================================================
resultado_pessoa = pipeline_homografia_mosaico(
    path_img1='imagens/pessoa_img1.jpg',
    path_img2='imagens/pessoa_img2.jpg',
    titulo_experimento='Experimento A.1 — Pessoa'
)

### 3.3 Parte 2A.2 — Experimento com Livro ou Caixa

Neste experimento, tiramos duas fotos de uma cena contendo um **livro** (ou caixa), com sobreposição de pelo menos 50% entre elas.

In [ ]:
# =====================================================================
# EXPERIMENTO A.2 — Livro / Caixa
# Substitua pelos nomes dos seus arquivos de imagem
# =====================================================================
resultado_livro = pipeline_homografia_mosaico(
    path_img1='imagens/livro_img1.jpg',
    path_img2='imagens/livro_img2.jpg',
    titulo_experimento='Experimento A.2 — Livro'
)

### 3.4 Parte 2B — Homografia e Mosaico em Tempo Real com Duas Webcams

Este código realiza a leitura simultânea de duas webcams, executando o pipeline de detecção SIFT, matching, estimação de homografia (RANSAC) e composição do mosaico em tempo real, exibindo os resultados em janelas de vídeo.

O script standalone correspondente está disponível em `stitching_webcam.py`.

In [ ]:
import cv2
import numpy as np


def stitching_webcam_realtime():
    cap1 = cv2.VideoCapture(0)
    cap2 = cv2.VideoCapture(2)

    if not cap1.isOpened() or not cap2.isOpened():
        print("Erro: Não foi possível abrir as webcams.")
        print("Verifique se duas webcams estão conectadas.")
        if cap1.isOpened():
            print("Usando webcam 1 para ambos os feeds (modo demonstração).")
            cap2 = cap1
        else:
            return

    sift = cv2.SIFT_create()
    bf = cv2.BFMatcher()

    print("Pressione 'q' para sair.")
    print("Pressione 's' para salvar um frame.")

    frame_count = 0

    while True:
        ret1, frame1 = cap1.read()
        ret2, frame2 = cap2.read()

        if not ret1 or not ret2:
            print("Erro ao capturar frame.")
            break

        frame1 = cv2.resize(frame1, (480, 360))
        frame2 = cv2.resize(frame2, (480, 360))

        gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        kp1, des1 = sift.detectAndCompute(gray1, None)
        kp2, des2 = sift.detectAndCompute(gray2, None)

        mosaico = None
        info_text = "Sem matches suficientes"

        if des1 is not None and des2 is not None and len(des1) > 2 and len(des2) > 2:
            matches_brutos = bf.knnMatch(des1, des2, k=2)

            bons_matches = []
            for m, n in matches_brutos:
                if m.distance < 0.75 * n.distance:
                    bons_matches.append(m)

            MIN_MATCH = 10

            if len(bons_matches) >= MIN_MATCH:
                src_pts = np.float32([kp1[m.queryIdx].pt for m in bons_matches]).reshape(-1, 1, 2)
                dst_pts = np.float32([kp2[m.trainIdx].pt for m in bons_matches]).reshape(-1, 1, 2)

                H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

                if H is not None:
                    inliers = int(np.sum(mask))
                    info_text = f"Matches: {len(bons_matches)} | Inliers: {inliers}"

                    # Correspondências (inliers)
                    img_matches = cv2.drawMatches(
                        frame1, kp1, frame2, kp2, bons_matches, None,
                        matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                        matchesMask=mask.ravel().tolist(), flags=0
                    )

                    # Mosaico em tempo real
                    width = frame1.shape[1] + frame2.shape[1]
                    height = max(frame1.shape[0], frame2.shape[0])
                    warped = cv2.warpPerspective(frame1, H, (width, height))
                    mosaico = warped.copy()
                    mosaico[0:frame2.shape[0], 0:frame2.shape[1]] = frame2

                    cv2.putText(img_matches, info_text, (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                    cv2.imshow('Correspondencias RANSAC', img_matches)

        if mosaico is not None:
            cv2.imshow('Mosaico Tempo Real', mosaico)
        else:
            side_by_side = np.hstack([frame1, frame2])
            cv2.putText(side_by_side, info_text, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            cv2.imshow('Mosaico Tempo Real', side_by_side)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            if mosaico is not None:
                fname = f'imagens/webcam_mosaico_{frame_count}.png'
                cv2.imwrite(fname, mosaico)
                print(f"Frame salvo: {fname}")
                frame_count += 1

    cap1.release()
    cap2.release()
    cv2.destroyAllWindows()


# Descomente a linha abaixo para executar (requer duas webcams):
# stitching_webcam_realtime()

## 4. Análise e Discussão

### 4.1 Análise Estatística das Características

**Cálculo da proporção de outliers:**

A porcentagem de outliers é calculada pela fórmula:

$$\%\text{Outliers} = \left(1 - \frac{\text{Inliers}}{\text{Total de Matches}}\right) \times 100$$

In [ ]:
# Análise estatística para ambos os experimentos
for nome, res in [('Pessoa', resultado_pessoa), ('Livro', resultado_livro)]:
    print(f"\n=== Experimento: {nome} ===")
    print(f"Total de matches (pós-Lowe): {res['total_matches']}")
    print(f"Inliers (RANSAC):            {res['inliers']}")
    print(f"Outliers:                     {res['outliers']}")
    print(f"% Outliers:                   {res['pct_outliers']:.2f}%")

**Interpretação:**

Os outliers podem ser causados por diversas características visuais do ambiente:

- **Texturas repetitivas:** padrões que se repetem (como azulejos, grades ou tecidos) geram descritores SIFT muito semelhantes em múltiplas posições, levando a correspondências incorretas.
- **Reflexos e superfícies especulares:** reflexos alteram a aparência local de forma diferente entre as duas capturas, criando falsos positivos.
- **Sombras e variações de iluminação:** regiões com sombras podem ter descritores distorcidos entre as imagens.
- **Objetos em movimento:** no caso do experimento com pessoa, movimentos entre as capturas geram correspondências espacialmente inconsistentes com a homografia do fundo estático.

No experimento com pessoa, espera-se uma porcentagem de outliers maior do que no experimento com livro/caixa, pois o corpo humano não é um plano rígido a homografia modela apenas transformações planares, e pontos na pessoa que se moveu violam essa suposição.

---

### 4.2 Álgebra Linear e Homografia — Comparação de Matrizes

**Por que $h_{22}$ costuma ser normalizado como 1.0?**

A matriz de homografia $H$ é definida em coordenadas homogêneas, onde a transformação é invariante a um fator de escala:

$$H \sim \lambda H, \quad \forall \lambda \neq 0$$

Isso significa que a matriz tem apenas 8 graus de liberdade, não 9. Para eliminar a ambiguidade de escala, convenciona-se normalizar o último elemento ($h_{22}$) como 1.0. Isso equivale a fixar a escala e tornar a representação única.

Geometricamente, $h_{22} = 1$ garante que pontos no infinito são tratados de forma consistente e que a escala da projeção é preservada para pontos próximos ao centro da imagem.

**Diferença nos elementos de translação:**

In [ ]:
# Comparação numérica das matrizes para ambos os experimentos
for nome, res in [('Pessoa', resultado_pessoa), ('Livro', resultado_livro)]:
    H_ls = res['H_ls']
    H_ransac = res['H_ransac']

    print(f"\n=== Experimento: {nome} ===")
    print(f"h22 (LS):     {H_ls[2,2]:.6f}")
    print(f"h22 (RANSAC): {H_ransac[2,2]:.6f}")

    diff_h02 = abs(H_ls[0, 2] - H_ransac[0, 2])
    diff_h12 = abs(H_ls[1, 2] - H_ransac[1, 2])

    print(f"\nDiferença absoluta na translação horizontal (h02): {diff_h02:.4f} pixels")
    print(f"Diferença absoluta na translação vertical   (h12): {diff_h12:.4f} pixels")

    print(f"\nMatriz H completa — Mínimos Quadrados:")
    print(H_ls)
    print(f"\nMatriz H completa — RANSAC:")
    print(H_ransac)

**Interpretação:**

A presença de outliers distorce os elementos de translação ($h_{02}$ e $h_{12}$) no método de Mínimos Quadrados Tradicional. Isso ocorre porque o método tenta minimizar o erro total de todos os pontos simultaneamente, inclusive os outliers. Correspondências incorretas "puxam" a solução para longe da transformação verdadeira, introduzindo translações espúrias.

Em cenários com muitos outliers, essa distorção pode ser severa o suficiente para tornar o mosaico completamente incoerente. O RANSAC, por outro lado, ignora os outliers durante a estimação, produzindo uma homografia que reflete fielmente a geometria real da cena.

---

### 4.3 Análise Geométrica e Robótica — Restrição de Modelo

**Número mínimo de pares de pontos para a homografia:**

A matriz de homografia $H$ ($3 \times 3$) possui 9 elementos, mas é definida a menos de escala, resultando em **8 graus de liberdade**. Cada par de pontos correspondentes $(x, y) \leftrightarrow (x', y')$ fornece **2 equações lineares** independentes (uma para cada coordenada).

Portanto, o número mínimo de pares é:

$$n_{\min} = \frac{8}{2} = 4 \text{ pares de pontos}$$

Com 4 pares de pontos, temos $4 \times 2 = 8$ equações para 8 incógnitas, determinando $H$ de forma única.

**Superfície sem texturas em robótica móvel:**

Se um drone focar em uma superfície completamente uniforme (sem texturas), o detector SIFT (ou qualquer detector baseado em gradientes) não encontrará keypoints, pois não há variações de intensidade para gerar extremos no espaço de escala. Sem keypoints, não há correspondências possíveis, e a matriz de homografia não pode ser calculada.

Isso tem consequências diretas na navegação:
- O sistema de odometria visual perde a referência, resultando em **drift** acumulativo na estimativa de posição.
- Em aplicações de SLAM visual, a perda de features causa falhas no rastreamento e na construção do mapa.
- Para contornar esse problema, drones reais combinam visão com sensores inerciais (IMU), GPS ou sensores LiDAR, garantindo continuidade na navegação mesmo sobre superfícies sem textura.

---

### 4.4 Qualidade do Processamento de Imagem — Análise de Artefatos Visuais

Na zona de junção do mosaico (`mosaico_ransac`), é comum observar artefatos visuais como:

- **Linha de corte visível:** a transição abrupta entre a imagem 2 (colada diretamente) e a imagem 1 projetada cria uma borda nítida.
- **Diferenças de iluminação:** as duas câmeras podem ter exposições diferentes, causando variações de brilho e cor na zona de emenda.
- **Efeito fantasma (ghosting):** objetos que se moveram entre as capturas aparecem duplicados ou translúcidos na sobreposição.

**Técnica de pós-processamento proposta — Multi-band Blending:**

Szeliski (2022, cap. 9) descreve a técnica de **Multi-band Blending** (Burt & Adelson, 1983) como solução para minimizar discrepâncias visuais na costura. O método opera da seguinte forma:

1. Decompõe cada imagem em uma pirâmide Laplaciana (bandas de frequência).
2. Para cada nível da pirâmide, realiza uma transição suave entre as imagens na zona de sobreposição, usando uma máscara de blending (tipicamente baseada na distância ao centro de cada imagem).
3. Reconstrói a imagem final a partir da pirâmide combinada.

A vantagem dessa abordagem é que variações de baixa frequência (iluminação global) são suavizadas em uma escala ampla, enquanto detalhes de alta frequência (bordas, texturas) são preservados e fundidos em uma escala mais fina. Isso elimina as linhas de corte e atenua diferenças de iluminação sem borrar a imagem final.

---

### 4.5 Aplicações no Trabalho Final — BlindDistance

O trabalho final do grupo é o **BlindDistance** ([repositório](https://github.com/MarioCerconvis/BlindDistance)), um sistema de tecnologia assistiva que utiliza visão computacional para auxiliar pessoas com deficiência visual a navegar pelo ambiente. O sistema emprega uma câmera 3D Orbbec Astra para capturar streams de cor e profundidade em tempo real, combinando detecção de obstáculos via YOLOv8 (`ultralytics`), mapeamento espacial de profundidade via OpenNI e feedback sonoro via `pyttsx3`/`pygame`.

As técnicas de alinhamento, homografia e costura de imagens estudadas neste laboratório possuem aplicações diretas e potenciais no BlindDistance:

- **Ampliação do campo de visão via Image Stitching:** a câmera Orbbec Astra possui um campo de visão limitado. Utilizando a homografia para alinhar frames consecutivos capturados durante o movimento do usuário, seria possível construir um **mosaico panorâmico do ambiente**, ampliando a percepção espacial além do campo instantâneo do sensor. Isso permitiria ao sistema manter consciência de obstáculos que já saíram do enquadramento atual.

- **Estabilização do mapa de profundidade:** o sistema atual (`main.py`) utiliza uma grade de amostragem (`grid_step_y=20, grid_step_x=20`) sobre o mapa de profundidade para detecção de proximidade. A estimação de homografia entre frames consecutivos permitiria **alinhar temporalmente os mapas de profundidade**, compensando tremores e movimentos bruscos do usuário, reduzindo falsos positivos na detecção de obstáculos próximos.

- **Detecção de degraus e desnível (Floor Drop):** o BlindDistance já implementa detecção de buracos/escadas usando média móvel exponencial da profundidade do chão (`floor_ema_alpha=0.05`). A homografia poderia melhorar essa detecção ao **alinhar a região do chão entre frames**, permitindo uma comparação direta pixel-a-pixel da profundidade antes e depois do movimento, tornando a detecção de variações abruptas de profundidade mais precisa e menos suscetível a mudanças de perspectiva.

- **Registro cor-profundidade robusto:** o sistema usa `IMAGE_REGISTRATION_DEPTH_TO_COLOR` do OpenNI para alinhar os streams. Em cenários onde o registro nativo falha ou é impreciso, a homografia estimada via features SIFT entre a imagem RGB e uma representação visual do mapa de profundidade poderia servir como **fallback de calibração**, garantindo que as bounding boxes do YOLOv8 correspondam corretamente às distâncias medidas.

- **SLAM visual para mapeamento do ambiente:** integrando a estimação de homografia frame-a-frame com os dados de profundidade da Orbbec, seria possível implementar um sistema básico de **odometria visual**, construindo um mapa 2D do ambiente ao redor do usuário. Isso permitiria ao BlindDistance fornecer orientações como "há uma parede a 2 metros à sua esquerda", melhorando a navegação em espaços fechados (Mur-Artal et al., 2015).

Essas extensões demonstram como as técnicas de homografia e alinhamento, aliadas ao sensor de profundidade já presente no BlindDistance, podem evoluir o sistema de alertas pontuais para uma compreensão espacial mais completa do ambiente.

## 5. Conclusões

Neste laboratório, estudamos e implementamos com sucesso o pipeline de alinhamento de imagens via homografia 2D:

1. **Fundamentação teórica**: Compreendemos os princípios matemáticos da homografia 2D, os métodos de estimação (Mínimos Quadrados e RANSAC) e o pipeline completo de Image Stitching.

2. **Feature Matching e Homografia**: Implementamos o pipeline completo do professor, incluindo:
   - Detecção de keypoints SIFT
   - Matching com BFMatcher e filtro de Lowe
   - Estimação de homografia via Mínimos Quadrados e RANSAC
   - Comparação quantitativa das matrizes e contagem de inliers/outliers

3. **Image Stitching**: Construímos mosaicos a partir de pares de imagens, verificando na prática a superioridade do RANSAC sobre o método de Mínimos Quadrados clássicos na presença de outliers.

4. **Tempo real**: Desenvolvemos um script para homografia e mosaico em tempo real com duas webcams, demonstrando a viabilidade do pipeline para aplicações interativas.

5. **Análise quantitativa**: Realizamos análise estatística dos outliers, comparação algébrica das matrizes de homografia, e discussão sobre restrições geométricas e artefatos visuais.

Os resultados demonstram que o RANSAC é essencial para estimação robusta de homografias em cenários reais, onde outliers são inevitáveis. As técnicas estudadas constituem a base para aplicações avançadas em robótica, mapeamento e realidade aumentada.

## 6. Referências

1. **Lowe, D. G.** (2004). "Distinctive Image Features from Scale-Invariant Keypoints." *International Journal of Computer Vision*, 60(2), pp. 91–110.

2. **Fischler, M. A.; Bolles, R. C.** (1981). "Random Sample Consensus: A Paradigm for Model Fitting with Applications to Image Analysis and Automated Cartography." *Communications of the ACM*, 24(6), pp. 381–395.

3. **Brown, M.; Lowe, D. G.** (2007). "Automatic Panoramic Image Stitching using Invariant Features." *International Journal of Computer Vision*, 74(1), pp. 59–73.

4. **Burt, P. J.; Adelson, E. H.** (1983). "A Multiresolution Spline with Application to Image Mosaics." *ACM Transactions on Graphics*, 2(4), pp. 217–236.

5. **Szeliski, R.** (2022). *Computer Vision: Algorithms and Applications*. 2nd ed. Springer.

6. **Mur-Artal, R.; Montiel, J. M. M.; Tardós, J. D.** (2015). "ORB-SLAM: A Versatile and Accurate Monocular SLAM System." *IEEE Transactions on Robotics*, 31(5), pp. 1147–1163.

7. **Hartley, R.; Zisserman, A.** (2003). *Multiple View Geometry in Computer Vision*. 2nd ed. Cambridge University Press.

8. **OpenCV Documentation** — Feature Matching + Homography to find Objects. Disponível em: <https://docs.opencv.org/5.0/py_tutorials/py_features/py_feature_homography/py_feature_homography.html>

9. **LearnOpenCV** — Homography Examples using OpenCV. Disponível em: <https://learnopencv.com/homography-examples-using-opencv-python-c/>

10. **GeeksForGeeks** — Image Stitching with OpenCV. Disponível em: <https://www.geeksforgeeks.org/computer-vision/image-stitching-with-opencv/>

11. **PyImageSearch** — OpenCV Panorama Stitching. Disponível em: <https://pyimagesearch.com/2016/01/11/opencv-panorama-stitching/>

---

**Nota sobre uso de IA Generativa:** Este relatório utilizou a ferramenta de IA Generativa Devin (Cognition AI) para auxiliar na formatação e organização do relatório. Todo o conteúdo final, análises, experimentos e interpretações foram revisados e validados pelos autores, conforme a Portaria CNPq 2664/2026.